Encode COCO GT boxes to DINO features

In [ ]:
import sys
from pathlib import Path

repo = Path("..").resolve()
sys.path.insert(0, str(repo))

In [ ]:
run_dir = Path("../outputs/notebook/encode_coco_gt_normalized").resolve()
run_dir.mkdir(parents=True, exist_ok=True)
print(run_dir)

In [ ]:
# DDETR detections from detector output
# cfg = {
#     "det_paths":   [repo / "outputs/notebook/detect/bboxes/detections.json"],
#     "ann_paths":   [],        # optional: assign GT labels via IoU
#     # "ann_paths": [repo / "data/OWDETR/VOC2007/Annotations/instances_train2017.json"],
#     "images_dir":  repo / "data/images",
#     "gt_assign_iou_threshold": 0.5,
#     "min_box_side": 2.0,

#     # Encoder
#     "enc_repo":     repo / "third_party/dinov3",
#     "enc_weights":  repo / "checkpoints/dinov3/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth",
#     "enc_model":    "dinov3_vitb16",
#     "enc_device":   "cuda",
#     "enc_image_size": 224,
#     "batch_size":   8,
# }

# COCO GT mode: build boxes directly from annotation JSON
cfg = {
    "det_paths":  [
        repo / "data/OWDETR/VOC2007/Annotations/instances_train2017.json",
        repo / "data/OWDETR/VOC2007/Annotations/instances_val2017.json",
    ],
    "ann_paths":   [],
    "images_dir":  repo / "data/OWDETR/VOC2007/JPEGImages",
    "min_box_side": 2.0,
    "enc_repo":     repo / "third_party/dinov3",
    "enc_weights":  repo / "checkpoints/dinov3/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth",
    "enc_model":    "dinov3_vitb16",
    "enc_device":   "cuda",
    "enc_image_size": 224,
    "batch_size":   8,
}

In [ ]:
from scripts.encode import stage_encode

stage_encode(cfg, run_dir)

In [ ]:
import json

saved = {k: [str(x) for x in v] if isinstance(v, list) else (str(v) if isinstance(v, Path) else v) for k, v in cfg.items()}
saved["feat_dir"] = str(run_dir)
(run_dir / "encode_config.json").write_text(json.dumps(saved, indent=2))
print(f"config saved -> {run_dir / 'encode_config.json'}")

In [ ]:
import json, random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

records = json.loads((run_dir / "records.json").read_text())
sample  = random.sample(records, min(5, len(records)))

fig, axes = plt.subplots(1, len(sample), figsize=(3 * len(sample), 3))
if len(sample) == 1:
    axes = [axes]

for ax, rec in zip(axes, sample):
    img = np.asarray(Image.open(rec["image_path"]).convert("RGB"))
    x1, y1, x2, y2 = [int(round(v)) for v in rec["box_xyxy"]]
    crop = img[y1:y2, x1:x2]
    ax.imshow(crop)
    label = rec.get("gt_category") or "?"
    score = rec.get("score")
    ax.set_title(f"{label}" + (f"\n{score:.2f}" if score else ""), fontsize=8)
    ax.axis("off")

fig.suptitle("5 random encoded crops", fontsize=10)
plt.tight_layout()
plt.show()
print(f"Total records: {len(records)}")